# Rag Assignment
Authors: Gianluca Amato and David Farrugia

--- short notebook description and overview of sections

## 1. Setup

### 1.1 Imports

This section imports all the required Python libraries used throughout the notebook and a fixed random seed is set for both Python’s `random` module and NumPy for reproducible results across multiple executions of the notebook.

In [2]:
import time
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

import torch
import requests
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (DPRQuestionEncoder, DPRQuestionEncoderTokenizer, DPRContextEncoder, DPRContextEncoderTokenizer)

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('punkt_tab')

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

c:\Users\david\Desktop\GitHub\ICS5111-MiningLargeScaleData_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### 1.2 Path Setup and Input Validation

This section is responsible for setting up the directory structure required by the notebook and validating the presence of essential input data files.

Output and temporary dataset directories are created automatically if they do not already exist, ensuring that files can be written without manual setup at all stages of the notebook. The use of `Pathlib` allows for clean and platform independent path management.

The input dataset directory and the expected CSV file are then defined. A helper function is initialised to validate the input CSV path by checking that the file exists, is a valid file (not a directory), and has the correct `.csv` extension.

This validation step helps catch configuration or file errors early, preventing silent failures or misleading results later

In [3]:
# Create Dataset Directories if they don't exist
DATA_OUTPUT_DIR = Path("./Datasets/Outputs")
DATA_TEMP_DIR = Path("./Datasets/TempDatasets")

for directory in [DATA_OUTPUT_DIR, DATA_TEMP_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

# Initialise Input Folder and Input Dataset Paths
DATA_INPUT_DIR = Path("./Datasets/Inputs")
FSA_PATH = DATA_INPUT_DIR / "FSA_data.csv"

def validate_csv_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path.resolve()}")
    if not path.is_file():
        raise ValueError(f"{label} exists but is not a file: {path.resolve()}")
    if path.suffix.lower() != ".csv":
        raise ValueError(f"{label} is not a CSV file: {path.resolve()}")
    print(f"[OK] {label}: {path.resolve()}")
    
validate_csv_path(FSA_PATH, "FSA Data")

[OK] FSA Data: C:\Users\david\Desktop\GitHub\ICS5111-MiningLargeScaleData_Project\Datasets\Inputs\FSA_data.csv


## 2. Data Processing

This section defines the core text preprocessing function used throughout the data processing stage of the pipeline.

An English stopword list is first initialised using the NLTK library. Stopwords are common function words (such as “the”, “and”, or “is”) that typically carry little discriminative value in term-based retrieval models and are therefore removed during sparse preprocessing.

A preprocessing function is then defined to normalise and filter text data. The function performs basic cleaning by converting text to lowercase, tokenising it into individual terms, removing stopwords, and discarding non-alphabetic tokens. This reduces noise while preserving content-bearing terms that are informative for sparse retrieval techniques such as TF–IDF or BM25.

An optional case is provided to retain four digit numeric tokens corresponding to years. This is particularly relevant in financial and economic datasets, where temporal information can play an important role in retrieval and interpretation.

The output of this function is a string of filtered tokens, which is later stored alongside the original raw text. This helps us maintaining both representations and allows the pipeline to apply different retrieval strategies using the appropriate format of data.


In [4]:
# Get English stop words set
stop_words = set(stopwords.words('english'))
    
# Function to preprocess text for sparse representation
def tokenize_and_filter(text, keep_years=False):
    # Guard against NaN / non-string values
    if not isinstance(text, str):
        return ""
    
    # Lowercase, tokenize, remove stopwords and non-alphabetic tokens
    tokens = word_tokenize(text.lower())
    
    filtered_tokens = []
    
    # Filter tokens
    for word in tokens:
        # keep alphabetic words
        if word.isalpha() and word not in stop_words:
            filtered_tokens.append(word)
            
            # optionally keep years (4-digit numbers)
        elif keep_years == True and word.isdigit() and len(word) == 4:
            filtered_tokens.append(word)
        
    # Join tokens back into a single string
    return " ".join(filtered_tokens)

#### 2.1 [Kaggle Financial Sentiment Analysis Dataset](https://www.kaggle.com/sbhatti/financial-sentiment-analysis)

This subsection processes the Kaggle Financial Sentiment Analysis dataset. The dataset is first loaded from disk and annotated with a source identifier to track its origin once multiple datasets are combined.

Each sentence is then preprocessed using the shared preprocessing function, producing a cleaned `processed_text` representation while preserving the original raw text. This dual representation allows different retrieval strategies to operate on the most appropriate form of the data.

A year extraction step is applied to the processed text by scanning for valid four-digit numeric tokens within a predefined range. When present, the extracted year is stored as structured metadata, enabling temporal filtering or analysis in later stages of the pipeline. This was mainly done since the 2nd dataset automatically returns a year column.

In [5]:
# Load dataset from disk
kaggle_dataset = pd.read_csv(FSA_PATH)

# Apply preprocessing to dataset for each sentence and reorder columns for better readability
kaggle_dataset["source"] = "kaggle"
kaggle_dataset["processed_text"] = kaggle_dataset["Sentence"].apply(tokenize_and_filter, keep_years=True)

# Create a year column based on processed text (if any year exists in the text)
def extract_year(text, min_year=2000, max_year=2025):
    tokens = text.split()
    
    for token in tokens:
        # Check if token is a 4-digit number
        if token.isdigit() and len(token) == 4:
            # Convert token to integer
            year = int(token)
            
            # Check if year is within valid range
            if min_year <= year <= max_year:
                return year
        
    return None

# Extract year from processed text
kaggle_dataset["year"] = kaggle_dataset["processed_text"].apply(extract_year)

# Rename columns and reorder for better readability
kaggle_dataset = kaggle_dataset.rename(columns={"Sentence": "text", "Sentiment": "sentiment"})
kaggle_dataset = kaggle_dataset[["source", "text", "processed_text", "year", "sentiment"]]

print("Kaggle FSA Processed Data Sample:")
display(kaggle_dataset.head(10))

# Save processed dataset to disk
KAGGLE_OUTPUT_PATH = DATA_TEMP_DIR / "kaggle_fsa_dataset.csv"
kaggle_dataset.to_csv(KAGGLE_OUTPUT_PATH, index=False)

Kaggle FSA Processed Data Sample:


,source,text,processed_text,year,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,NaN,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,NaN,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,2010.0,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,NaN,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,NaN,neutral
5,kaggle,$SPY wouldn't be surprised to see a green close,spy would surprised see green close,NaN,positive
6,kaggle,Shell's $70 Billion BG Deal Meets Shareholder ...,shell billion bg deal meets shareholder skepti...,NaN,negative
7,kaggle,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,ssh communications security corp stock exchang...,2008.0,negative
8,kaggle,Kone 's net sales rose by some 14 % year-on-ye...,kone net sales rose first nine months 2008,2008.0,positive
9,kaggle,The Stockmann department store will have a tot...,stockmann department store total floor space s...,NaN,neutral


### 2.2 [World Bank Open Data](https://data.worldbank.org/)

This subsection retrieves structured economic indicators from the World Bank Open Data API. A base API endpoint is defined and used to fetch indicator values for selected countries over a specified time range.

A helper function is implemented to handle data retrieval. For each country, the function creates the appropriate API request, specifies the desired date range, and requests the data in JSON format. Error handling is applied to detect and halt execution in the case of invalid or failed requests.

Returned records are then parsed and transformed into a tabular format containing the country name, year, and indicator value. Requests are issued sequentially with a short delay between calls to avoid exceeding API rate limits. The output is a unified DataFrame containing all retrieved records.

> **[!NOTE]**
> **The API connection can occasionally timeout. If this happens please rerun the notebook again** 

In [6]:
WB_API_URL = "https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"

# Function to fetch data from World Bank API
def fetch_api_data(indicator, countries, start_year, end_year, per_page=2000, rate_limit_delay=0.2):
    '''
    This fuction fetches data from World Bank API for given indicator and countries within specified year range.
    
    Parameters:
    - indicator (str): World Bank indicator code
    - contries (list): List of country codes
    - start_year (int): Start year for data
    - end_year (int): End year for data
    - per_page (int): Number of records per page
    - rate_limit_delay (float): Delay between requests to avoid rate limiting
    
    Returns:
    - DataFrame with columns ['country', 'year', 'value']
    '''
    all_records = []
    
    # Loop through countries one by one
    for country in countries:
        # Construct API URL
        url = WB_API_URL.format(
            country=country,
            indicator=indicator,
        )
        
        # Set query parameters
        params = {
            "date": f"{start_year}:{end_year}",
            "format": "json",
            "per_page": per_page,
        }
        
        # Make API request
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status() # Raise error for bad responses
        data = response.json()
    
        if len(data) < 2 or not data[1]:
            # No returned data for this country, skip
            time.sleep(rate_limit_delay)
            continue
        
        # Append records for this country
        for entry in data[1]:
            all_records.append({
                "country": entry["country"]["value"],
                "year": entry["date"],
                "value": entry["value"]
            })
    
        # Wait before next request so we don't spam the API
        time.sleep(rate_limit_delay)
    
    return pd.DataFrame(all_records)

This subsection retrieves economic indicators from the World Bank Open Data API for a predefined set of countries and years. A small collection of indicators relevant to financial and economic analysis is specified, along with the target countries and time range.

For each indicator, data is fetched for all countries using the previously defined API retrieval function. The results are annotated with a descriptive indicator name and combined into a single dataset covering multiple economic dimensions. The dataset is then labelled with a source identifier and its columns are reordered into a consistent structure.

In [7]:
# Congifure parameters for data fetching
COUNTRIES = {
    "MLT": "Malta",
    "ITA": "Italy",
    "DEU": "Germany",
    "FRA": "France",
    "ESP": "Spain",
    "PRT": "Portugal",
    "IRL": "Ireland",
    "NLD": "Netherlands",
}

INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "GDP growth (%)",
    "FP.CPI.TOTL.ZG": "Inflation rate (annual %)",
    "GC.DOD.TOTL.GD.ZS": "Government debt (% of GDP)",
    "SL.EMP.TOTL.ZS": "Employment rate (% of total labor force)",
    "SL.UEM.TOTL.ZS": "Unemployment rate (% of total labor force)"
}

START_YEAR = 2000
END_YEAR = 2025

# Fetch data
wb_dataset = []

# For each indicator, fetch data and append to dataset
for indicator_code, indicator_name in INDICATORS.items():
    df = fetch_api_data(
        indicator=indicator_code,
        countries=list(COUNTRIES.keys()),
        start_year=START_YEAR,
        end_year=END_YEAR
        # per_page = 2000,          # Parameters already set to default values in function
        # rate_limit_delay = 0.2    # Parameters already set to default values in function
    )

    df["indicator_name"] = indicator_name
    wb_dataset.append(df)

# Combine all indicator data into a single DataFrame
wb_dataset = pd.concat(wb_dataset, ignore_index=True)
wb_dataset["source"] = "wb_open_data"

# Reorder columns for better readability
wb_dataset = wb_dataset[["source", "country", "year", "indicator_name", "value"]]

print("World Bank Economic Indicators Data Sample:")
display(wb_dataset.head(10))

World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value
0,wb_open_data,Malta,2024,GDP growth (%),6.799903
1,wb_open_data,Malta,2023,GDP growth (%),10.621519
2,wb_open_data,Malta,2022,GDP growth (%),2.483100
3,wb_open_data,Malta,2021,GDP growth (%),13.411704
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283
5,wb_open_data,Malta,2019,GDP growth (%),4.085056
6,wb_open_data,Malta,2018,GDP growth (%),7.189215
7,wb_open_data,Malta,2017,GDP growth (%),12.971342
8,wb_open_data,Malta,2016,GDP growth (%),4.078004
9,wb_open_data,Malta,2015,GDP growth (%),9.620703


### 2.3 Formatting World Bank API Output

Since retrieval models operate on text rather than raw numerical tables, each World Bank data row is converted into a natural-language sentence.

For example:
> **“In 2024, Malta’s GDP growth (%) was 6.79.”**

This transformation allows for structured indicators to be indexed and retrieved in the same way as unstructured text documents.

In [8]:
# Create natural language representation for each World Bank data row
def wb_row_to_text(row):
    return f"In {row['year']}, {row['country']}'s {row['indicator_name']} was {row['value']:.6f}."

wb_dataset["text"] = wb_dataset.apply(wb_row_to_text, axis=1)
display(wb_dataset.head(10))

,source,country,year,indicator_name,value,text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903."
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519."
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100."
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704."
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283."
5,wb_open_data,Malta,2019,GDP growth (%),4.085056,"In 2019, Malta's GDP growth (%) was 4.085056."
6,wb_open_data,Malta,2018,GDP growth (%),7.189215,"In 2018, Malta's GDP growth (%) was 7.189215."
7,wb_open_data,Malta,2017,GDP growth (%),12.971342,"In 2017, Malta's GDP growth (%) was 12.971342."
8,wb_open_data,Malta,2016,GDP growth (%),4.078004,"In 2016, Malta's GDP growth (%) was 4.078004."
9,wb_open_data,Malta,2015,GDP growth (%),9.620703,"In 2015, Malta's GDP growth (%) was 9.620703."


The generated natural language sentences are further processed using the same preprocessing function used for the Kaggle dataset.

In this case, year tokens are retained to support time-specific queries (e.g., “GDP growth in 2020”). This produces a sparse representation suitable for keyword-based retrieval while preventing the loss of temporal information.

In [9]:
wb_dataset["processed_text"] = wb_dataset["text"].apply(tokenize_and_filter, keep_years=True)
display(wb_dataset.head(10))

# Save fetched data to CSV
WBF_OUTPUT_PATH = DATA_TEMP_DIR / "wb_economic_indicators_dataset.csv"
wb_dataset.to_csv(WBF_OUTPUT_PATH, index=False)

,source,country,year,indicator_name,value,text,processed_text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903.",2024 malta gdp growth
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519.",2023 malta gdp growth
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100.",2022 malta gdp growth
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704.",2021 malta gdp growth
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283.",2020 malta gdp growth
5,wb_open_data,Malta,2019,GDP growth (%),4.085056,"In 2019, Malta's GDP growth (%) was 4.085056.",2019 malta gdp growth
6,wb_open_data,Malta,2018,GDP growth (%),7.189215,"In 2018, Malta's GDP growth (%) was 7.189215.",2018 malta gdp growth
7,wb_open_data,Malta,2017,GDP growth (%),12.971342,"In 2017, Malta's GDP growth (%) was 12.971342.",2017 malta gdp growth
8,wb_open_data,Malta,2016,GDP growth (%),4.078004,"In 2016, Malta's GDP growth (%) was 4.078004.",2016 malta gdp growth
9,wb_open_data,Malta,2015,GDP growth (%),9.620703,"In 2015, Malta's GDP growth (%) was 9.620703.",2015 malta gdp growth


### 2.4 Combining Datasets

In [10]:
print("Kaggle FSA Processed Data Sample:")
display(kaggle_dataset.head())
print("World Bank Economic Indicators Data Sample:")
display(wb_dataset.head())

Kaggle FSA Processed Data Sample:


,source,text,processed_text,year,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,NaN,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,NaN,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,2010.0,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,NaN,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,NaN,neutral


World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value,text,processed_text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903.",2024 malta gdp growth
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519.",2023 malta gdp growth
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100.",2022 malta gdp growth
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704.",2021 malta gdp growth
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283.",2020 malta gdp growth


Before merging the datasets, empty metadata fields are added to the Kaggle dataset to ensure schema consistency across both data sources.

Both datasets are then reordered to follow a shared column structure:

- `source`
- `text`
- `processed_text`
- `country`
- `year`
- `indicator_name`
- `value`

The aligned datasets are concatenated into a single unified table, forming the final dataset used throughout the RAG pipeline. A sample of the merged data is displayed to verify correct integration.

The resulting dataset is saved to disk as a CSV file and represents the complete RAG corpus for this project, combining unstructured textual data with structured economic metadata from multiple sources.


In [11]:
# In order to keep metadata consistent across both datasets, we add empty columns to the Kaggle dataset
kaggle_dataset["country"] = None
kaggle_dataset["indicator_name"] = None
kaggle_dataset["value"] = None

# Final reordering of columns
kaggle_dataset = kaggle_dataset[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

wb_dataset = wb_dataset[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

# Combine both datasets into final RAG dataset
final_rag_dataset = pd.concat([kaggle_dataset, wb_dataset], ignore_index=True)

print("Final RAG Dataset Sample:")
display(final_rag_dataset.head(20))

# Save final RAG dataset to CSV
RAG_OUTPUT_PATH = DATA_OUTPUT_DIR / "final_rag_dataset.csv"
final_rag_dataset.to_csv(RAG_OUTPUT_PATH, index=False)

Final RAG Dataset Sample:


C:\Users\david\AppData\Local\Temp\ipykernel_18896\3269066736.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_rag_dataset = pd.concat([kaggle_dataset, wb_dataset], ignore_index=True)


,source,text,processed_text,country,year,indicator_name,value
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,None,NaN,None,NaN
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,None,NaN,None,NaN
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,None,2010.0,None,NaN
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,None,NaN,None,NaN
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,None,NaN,None,NaN
5,kaggle,$SPY wouldn't be surprised to see a green close,spy would surprised see green close,None,NaN,None,NaN
6,kaggle,Shell's $70 Billion BG Deal Meets Shareholder ...,shell billion bg deal meets shareholder skepti...,None,NaN,None,NaN
7,kaggle,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,ssh communications security corp stock exchang...,None,2008.0,None,NaN
8,kaggle,Kone 's net sales rose by some 14 % year-on-ye...,kone net sales rose first nine months 2008,None,2008.0,None,NaN
9,kaggle,The Stockmann department store will have a tot...,stockmann department store total floor space s...,None,NaN,None,NaN


## 3. Getting to know the data

This section provides an exploratory analysis of the final RAG dataset, with the main aim being to understand the structure and contents of the dataset before applying retrieval and generation methods.

### 3.1 Distribution of Data Sources

This subsection looks at the distribution of records across the different data sources included in the final dataset. A histogram is used to visualise the relative contribution of each source, helping to identify any potential imbalance between sources.

In [12]:
fig = px.histogram(
    final_rag_dataset,
    x="source",
    color="source",
    text_auto=True,
    title="Distribution of Data Sources",
    labels={"source": "Data Source"},
    color_discrete_sequence=["#AEC6CF", "#FF6961"]  # pastel blue & pastel pink
)

fig.show()

### 3.2 Missing Values per Column

This subsection analyses missing values across all columns in the dataset. The total number of missing entries per column is calculated and visualised to highlight which fields are sparsely populated.

This analysis helps distinguish between expected missing values (e.g. structured metadata not applicable to all records) and potential data quality issues that may require attention in later stages.

In [13]:
missing_df = final_rag_dataset.isna().sum().reset_index()
missing_df = missing_df[missing_df[0] > 0].sort_values(by=0, ascending=True)

if not missing_df.empty:
    missing_df.columns = ["column", "missing_count"]

    fig = px.bar(
        missing_df,
        x="column",
        y="missing_count",
        text_auto=True,
        title="Missing Values per Column",
        labels={"missing_count": "Number of Missing Values"},
        color="column",
        color_discrete_sequence=[
            "#AEC6CF",  # pastel blue
            "#FF6961",  # pastel pink
            "#C1E1C1",  # pastel green
            "#D7BDE2",  # pastel purple
            "#FFF2CC",  # pastel yellow
            "#FADADD",  # pastel peach
            "#D7B4F3"   # pastel lavender
        ]
    )

    fig.show()

### 3.3 Temporal Coverage (Year Distribution)

This subsection explores the temporal coverage of the dataset by analysing the distribution of available year metadata. Records without a year value are excluded from this analysis.

A histogram with an overlaid density curve is used to illustrate how data points are distributed over time. Summary statistics, including the mean, median, and mode, are visualised to provide additional insight into the central tendency and temporal concentration of the dataset.

As show by the plot, the data is mostly from 2005-2010, peaking around 2008–2009. The mean, median, and mode are close, showing that the data is well-centred in that period, with fewer records in earlier and later years. This means the dataset is temporally imbalanced and mainly reflects the late 2000s.

In [14]:
data = final_rag_dataset.dropna(subset=["year"])
years = data["year"].astype(int)

# Histogram
fig = px.histogram(
    data,
    x="year",
    nbins=30,
    histnorm="probability density",
    title="Distribution of Years with Density Curve",
    labels={"year": "Year"}
)


fig.update_traces(marker_color="#AEC6CF")

# KDE line
kde = gaussian_kde(years)
x = np.linspace(years.min(), years.max(), 1000)

fig.add_trace(
    go.Scatter(
        x=x,
        y=kde(x),
        mode="lines",
        name="Density Curve",
        line=dict(color="#FF6961", width=2)  # pastel pink
    )
)

# Adding mean, median, mode lines
stats = {
    "Mean": (years.mean(), "green"),
    "Median": (years.median(), "blue"),
    "Mode": (years.mode().iloc[0], "orange"),
}

for label, (value, color) in stats.items():
    fig.add_vline(
        x=value,
        line=dict(color=color, dash="dash"),
        name=f"{label} ({value:.1f})",
        showlegend=True
    )

fig.show()

### 3.4 Text Length Distribution

This subsection analyses the distribution of raw text lengths across the dataset. Text length is measured as the number of words per document.

A probability density histogram and kernel density estimate are used to characterise the overall shape of the distribution. Mean, median, and mode values are included to highlight typical document lengths and identify potential outliers or extreme cases.

The majority of texts are short, with the highest frequency around 10 words. The mean (~19.6) is higher than the median (18), indicating a right-skewed distribution caused by a smaller number of longer texts

In [15]:
temp = final_rag_dataset.copy()
temp["text_length"] = final_rag_dataset["text"].str.split().str.len()

fig = px.histogram(
    temp,
    x="text_length",
    nbins=50,
    histnorm="probability density",
    title="Distribution of Text Lengths",
    labels={"text_length": "Number of Words"}
)

fig.update_traces(marker_color="#AEC6CF")

# KDE line
kde = gaussian_kde(temp["text_length"])
x = np.linspace(temp["text_length"].min(), temp["text_length"].max(), 1000)

fig.add_trace(
    go.Scatter(
        x=x,
        y=kde(x),
        mode="lines",
        name="Density Curve",
        line=dict(color="#FF6961", width=2)  # pastel pink
    )
)

# Adding mean, median, mode lines
stats = {
    "Mean": (temp["text_length"].mean(), "green"),
    "Median": (temp["text_length"].median(), "blue"),
    "Mode": (temp["text_length"].mode().iloc[0], "orange"),
}

for label, (value, color) in stats.items():
    fig.add_vline(
        x=value,
        line=dict(color=color, dash="dash"),
        name=f"{label} ({value:.1f})",
        showlegend=True
    )

fig.show()

### 3.5 Processed Text Length Distribuution

This subsection examines the length distribution of the processed text representation. As with the raw text analysis, text length is measured in terms of word count.

Comparing this distribution with the raw text length provides insight into the impact of preprocessing steps such as stopword removal and token filtering, and helps assess how much textual compression is introduced by the sparse preprocessing pipeline.

Processed texts are significantly shorter than the raw texts, with most entries containing 7–10 words. The mean (~10.4) is slightly higher than the median (9), showing a right-skewed distribution caused by a small number of longer processed texts. This is primarily due to the effect of stopword removal and token filtering, which substantially reduces text length.

In [16]:
temp = final_rag_dataset.copy()
temp["pt_length"] = final_rag_dataset["processed_text"].str.split().str.len()

fig = px.histogram(
    temp,
    x="pt_length",
    nbins=50,
    histnorm="probability density",
    title="Distribution of Processed Text Lengths",
    labels={"pt_length": "Number of Words"}
)

fig.update_traces(marker_color="#AEC6CF")

# KDE line
kde = gaussian_kde(temp["pt_length"])
x = np.linspace(temp["pt_length"].min(), temp["pt_length"].max(), 1000)

fig.add_trace(
    go.Scatter(
        x=x,
        y=kde(x),
        mode="lines",
        name="Density Curve",
        line=dict(color="#FF6961", width=2)  # pastel pink
    )
)

# Adding mean, median, mode lines
stats = {
    "Mean": (temp["pt_length"].mean(), "green"),
    "Median": (temp["pt_length"].median(), "blue"),
    "Mode": (temp["pt_length"].mode().iloc[0], "orange"),
}

for label, (value, color) in stats.items():
    fig.add_vline(
        x=value,
        line=dict(color=color, dash="dash"),
        name=f"{label} ({value:.1f})",
        showlegend=True
    )

fig.show()

## 4. Sparse Retrieval Methods

In [17]:
#Calculating BM25 Score

def termFrequency(term, document):
    termCount = document.count(term)
    docLength = len(document)
    
    return (termCount / (termCount + k*(1 - b + b * (docLength / avgDocLength))))

def inverseDocFrequency(term):
    docsContainingTerm = final_rag_dataset['processed_text'].str.contains(term).sum()

    return math.log((totalCorpLength - docsContainingTerm + 0.5) / (docsContainingTerm + 0.5))

def bm25Score(query, document):
    bmScore = 0
    for term in query:
        tf = termFrequency(term, document)
        idf = inverseDocFrequency(term)
        bmScore += tf*idf
    return bmScore

k = 1.2
b = 0.75
avgDocLength = final_rag_dataset['processed_text'].str.split().str.len().mean()
totalCorpLength = len(final_rag_dataset)

query = "euro"

cleanedQuery = ''.join(char for char in query if char.isalnum() or char.isspace())
processedQuery = word_tokenize(cleanedQuery.lower())
for doc in final_rag_dataset.itertuples():
    processedDoc = doc[3].split()
    score = bm25Score(processedQuery, processedDoc)
    if score != 0:
        print(score)


1.1334472286033652
1.0687139632007057
1.1334472286033652
1.8694885537259323
1.6793156317032973
1.8915000163088416
1.246720198146762
1.5410209146550031
1.5099344480444963
1.1334472286033652
1.5410209146550031
1.5581536488789647
1.4513781769293403
1.438383217233785
1.5581536488789647
1.6793156317032973
1.6071987964663679
1.5734143018123925
1.5734143018123925
1.6258435312381332
1.6258435312381332
0.9843839812952033
1.6071987964663679
1.4513781769293403
1.717856722732717
1.3851472342924696
1.5581536488789647
1.438383217233785
1.699681728338957
1.5734143018123925
1.3851472342924696
0.9843839812952033
1.758208441195602
1.438383217233785
0.9591556594701466
1.5734143018123925
1.717856722732717
1.5581536488789647
1.6424659749105859
1.438383217233785
1.168846432185882
1.5581536488789647
1.5581536488789647
1.4513781769293403
1.438383217233785
1.1001291678991065
1.6258435312381332
1.0687139632007057
1.5734143018123925
2.046659348926504
1.4958748439003897
1.6424659749105859
1.8005014488692659
1.438

## 5. Dense Retrieval Methods

This section, impliments two **dense retrieval approaches** for information retrieval in a RAG pipeline. For **sentence-transformers**, the same feature space is used for both the database and user queries. While for **Dense Passage Retrieval (DPR)** separate neural encoders are used for queries and documents to perform semantic retrieval.

### 5.1 Sentence-Transformers

This subsection implements dense retrieval using Sentence-BERT (SBERT).
Documents and queries are embedded into the same feature space, and cosine similarity is used to retrieve the most relevant passages.

In [18]:
# Make sure 'text' column exists
if not "text" in final_rag_dataset.columns:
    raise ValueError("!!! Text column not found in RAG dataset !!!")

# Reset index to create a unique ID column for documents
documents = final_rag_dataset.sort_index().reset_index()
doc_texts = documents["text"].tolist()

# Rename index to id
documents = documents.rename(columns={"index": "id"})
doc_ids = documents["id"].tolist()

display(documents.head())

,id,source,text,processed_text,country,year,indicator_name,value
0,0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,None,NaN,None,NaN
1,1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,None,NaN,None,NaN
2,2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,None,2010.0,None,NaN
3,3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,None,NaN,None,NaN
4,4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,None,NaN,None,NaN


The document embeddings are generated using the `encode` method with parameters. A batch size of 16 is used as a safe compromise between memory consumption and processing speed, while progress tracking is enabled to provide feedback during long encoding operations. 

The embeddings are converted to NumPy arrays for easier processing and compatibility with similarity functions, while normalisation is applied so that cosine similarity so that similarity is based on meaning rather than vector size.


In [19]:
# Load SentenceTransformer model
# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode all document texts into normalized embeddings
doc_embeddings_st = model.encode(
    doc_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 416/416 [00:34<00:00, 12.11it/s]


This function implements semantic document retrieval using SBERT embeddings and cosine similarity. It encodes the given query into the same feature space as the documents, computes similarity scores against all stored document embeddings, and returns the top-ranked results.


In [20]:
def st_retrieve(model, query, top_k = 5):
    """
    Retrieve top_k documents using SBERT embeddings + cosine similarity.

    Parameters:
        query (str): user query
        top_k (int): number of documents to retrieve

    Returns:
        DataFrame containing top_k rows with similarity scores.
    """
    # Encode and normalize query embedding (same settings as corpus embeddings)
    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Compute cosine similarity between query and all document embeddings
    scores = cosine_similarity(query_embedding.reshape(1, -1), doc_embeddings_st)[0]
    scores = np.round((scores * 100), 3)

    # Select top-k document indices based on similarity
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Build a result table for inspection
    results = documents.iloc[top_indices][["id", "text", "source", "year"]].copy()
    results["similarity_score"] = scores[top_indices]
    results["rank"] = np.arange(1, len(results) + 1)

    return results.reset_index(drop=True)

This cell evaluates retrieval function using a small set of example queries related to economic topics.


In [21]:
queries = [
    "How did inflation change in Italy after 2020?",
    "Investors are worried about rising inflation amid political uncertainty",
    "What happened to unemployment in Malta after COVID?"
]

print("Sentence-Transformer Retrieval Results:")
for q in queries:
   st_results = st_retrieve(model, q, top_k=5)
   display(st_results)

Sentence-Transformer Retrieval Results:


,id,text,source,year,similarity_score,rank
0,6071,"In 2020, Italy's Inflation rate (annual %) was...",wb_open_data,2020,80.584999,1
1,6070,"In 2021, Italy's Inflation rate (annual %) was...",wb_open_data,2021,79.055000,2
2,6069,"In 2022, Italy's Inflation rate (annual %) was...",wb_open_data,2022,77.825996,3
3,6068,"In 2023, Italy's Inflation rate (annual %) was...",wb_open_data,2023,76.932999,4
4,6067,"In 2024, Italy's Inflation rate (annual %) was...",wb_open_data,2024,76.588997,5


,id,text,source,year,similarity_score,rank
0,4224,Small investors have voiced fears that the sha...,kaggle,NaN,50.042999,1
1,2274,Small investors have voiced fears that the sha...,kaggle,NaN,50.042999,2
2,871,The ECB can mainly target inflation .,kaggle,NaN,42.219002,3
3,3478,Aktia forecasts Finland 's inflation at 1.1 % ...,kaggle,2010.0,40.896000,4
4,2316,"Aviva, M&G suspend property funds as investors...",kaggle,NaN,39.071999,5


,id,text,source,year,similarity_score,rank
0,6454,"In 2012, Malta's Unemployment rate (% of total...",wb_open_data,2012,68.432999,1
1,6443,"In 2023, Malta's Unemployment rate (% of total...",wb_open_data,2023,68.154999,2
2,6459,"In 2007, Malta's Unemployment rate (% of total...",wb_open_data,2007,67.528000,3
3,6464,"In 2002, Malta's Unemployment rate (% of total...",wb_open_data,2002,67.344002,4
4,6456,"In 2010, Malta's Unemployment rate (% of total...",wb_open_data,2010,66.974998,5


### 5.2 DPR

This cell loads the pretrained DPR question and context encoders. These models are used to independently encode queries and documents into a shared feature space suitable for dense retrieval.


In [22]:
# Question encoder (for queries)
# https://huggingface.co/facebook/dpr-question_encoder-single-nq-base
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)
q_encoder = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)

# Context encoder (for documents)
# https://huggingface.co/facebook/dpr-ctx_encoder-single-nq-base
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
)
ctx_encoder = DPRContextEncoder.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
)

display(q_encoder.eval())
display(ctx_encoder.eval())

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRCon

DPRQuestionEncoder(
  (question_encoder): DPREncoder(
    (bert_model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_feature

DPRContextEncoder(
  (ctx_encoder): DPREncoder(
    (bert_model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768,

This function encodes all documents using the DPR context encoder in batches, producing dense vector representations that will later be used for similarity-based retrieval. The encoding function is then used to the full dataset, generating the embeddings required for retrieval.

In [23]:
def encode_documents_dpr(texts, batch_size=16):
    # List to store embeddings for all document batches
    embeddings = []

    # Process documents in batches to control memory usage
    for i in range(0, len(texts), batch_size):
        # Select a batch
        batch = texts[i:i + batch_size]

        # Tokenize the batch using the DPR context tokenizer
        inputs = ctx_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        # Disable gradient computation since this is inference, not training
        with torch.no_grad():
            outputs = ctx_encoder(**inputs)
            batch_embeddings = outputs.pooler_output

        # Store embeddings of the current batch
        embeddings.append(batch_embeddings)

    # Concatenate all batch embeddings into a single tensor
    return torch.cat(embeddings, dim=0)

# Encode all document texts into DPR embeddings
doc_embeddings_dpr = encode_documents_dpr(doc_texts)

This function performs document retrieval using DPR by encoding the query with the question encoder and computing dot-product similarity against the document embeddings to rank the most relevant results.

In [24]:
def dpr_retrieve(query, top_k=5):
    """
    Dense Passage Retrieval using DPR question/context encoders
    and dot-product similarity.
    """
    # Tokenize the query for the DPR question encoder
    inputs = q_tokenizer(
        query,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    # Encode the query into a dense vector representation
    with torch.no_grad():
        query_embedding = q_encoder(**inputs).pooler_output

    # Dot-product similarity (standard DPR scoring)
    scores = torch.matmul(
        query_embedding,
        doc_embeddings_dpr.T
    ).squeeze(0).numpy()

    # Select the indices of the top-k highest scoring documents
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Build a results table with relevant document information
    results = documents.iloc[top_indices][["id", "text", "source", "year"]].copy()
    results["similarity_score"] = np.round(scores[top_indices], 3)
    results["rank"] = np.arange(1, len(results) + 1)

    return results.reset_index(drop=True)


Utilising the same small set of example queries related to economic topics.

```ptyhon
queries = [
    "How did inflation change in Italy after 2020?",
    "Investors are worried about rising inflation amid political uncertainty",
    "What happened to unemployment in Malta after COVID?"
]
```

In [25]:
print("DPR Retrieval Results:")
for q in queries:
   dpr_results = dpr_retrieve(q, top_k=5)
   display(dpr_results)

DPR Retrieval Results:


,id,text,source,year,similarity_score,rank
0,6072,"In 2019, Italy's Inflation rate (annual %) was...",wb_open_data,2019,87.013000,1
1,6070,"In 2021, Italy's Inflation rate (annual %) was...",wb_open_data,2021,86.711998,2
2,6073,"In 2018, Italy's Inflation rate (annual %) was...",wb_open_data,2018,86.518997,3
3,6071,"In 2020, Italy's Inflation rate (annual %) was...",wb_open_data,2020,86.325996,4
4,6069,"In 2022, Italy's Inflation rate (annual %) was...",wb_open_data,2022,86.323997,5


,id,text,source,year,similarity_score,rank
0,3030,U.S. Debt Lures Schroders as ECB Depresses Rates,kaggle,NaN,74.690002,1
1,6094,"In 2022, Germany's Inflation rate (annual %) w...",wb_open_data,2022,73.719002,2
2,6092,"In 2024, Germany's Inflation rate (annual %) w...",wb_open_data,2024,73.225998,3
3,6093,"In 2023, Germany's Inflation rate (annual %) w...",wb_open_data,2023,73.147003,4
4,6069,"In 2022, Italy's Inflation rate (annual %) was...",wb_open_data,2022,72.749001,5


,id,text,source,year,similarity_score,rank
0,6449,"In 2017, Malta's Unemployment rate (% of total...",wb_open_data,2017,74.606003,1
1,6464,"In 2002, Malta's Unemployment rate (% of total...",wb_open_data,2002,74.291000,2
2,6459,"In 2007, Malta's Unemployment rate (% of total...",wb_open_data,2007,74.265999,3
3,6247,"In 2019, Malta's Government debt (% of GDP) wa...",wb_open_data,2019,74.019997,4
4,6447,"In 2019, Malta's Unemployment rate (% of total...",wb_open_data,2019,74.003998,5


---
> *This project was developed as part of the **ICS5111** course at the University of Malta.*